## 1. 核心数据结构：张量 (Tensor)

In [ ]:
import torch

# 从列表创建
data = [[1, 2], [3, 4]]
tensor = torch.tensor(data)

# 创建全0、全1或随机张量
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
random = torch.rand(2, 3)  # 均匀分布

print(f"张量形状: {tensor.shape}")
print(f"数据类型: {tensor.dtype}")
print(f"所在设备: {tensor.device}")  # cpu 或 cuda:0


In [ ]:
zeros

In [ ]:
ones

In [ ]:
random

## 2. 自动求导 (Autograd)

In [ ]:
# 需要求导的张量
x = torch.tensor([2.0], requires_grad=True)

# 进行操作
y = x ** 2 + 3 * x + 1

# 反向传播，自动计算梯度 dy/dx
y.backward()

# 打印梯度
print(x.grad)  # 输出: tensor([7.]) 因为 2*x + 3 = 7

## 3. 构建神经网络 (nn.Module)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):

        """
        input_size：输入数据的特征维度。例如，处理 MNIST 手写数字（28x28像素的图片）时，输入是 784 (28*28)。
        hidden_size：隐藏层的神经元数量，比如 128。这是一个超参数，决定了网络的容量。越大模型能力越强，但也更容易过拟合且计算更慢。
        num_classes：输出类别的数量。对于 MNIST 数字识别（0-9），输出就是 10 类。
        
        """
        super(SimpleNet, self).__init__()
        
        """
        含义：调用父类 nn.Module 的构造函数。
        为什么必须写：这行代码必须写在 __init__ 的开头。它负责执行 nn.Module 基类内部的初始化工作，比如初始化内部的数据结构来存储网络层和参数。
        如果没有这一行，PyTorch 将无法正确识别你的网络结构。
        """


        # 定义网络层


        # 全连接层1
        """
        nn.Linear 内部执行的计算是：output = input @ weight.T + bias。
        input：形状为 (batch_size, input_size)
        weight：形状为 (hidden_size, input_size) 的权重矩阵
        bias：形状为 (hidden_size) 的偏置向量
        output：形状为 (batch_size, hidden_size)
        """
        
        self.fc1 = nn.Linear(input_size, hidden_size)   


        # 激活函数
        """
        含义：实例化一个 ReLU (Rectified Linear Unit) 激活函数，并保存为网络的一个属性。
        作用：ReLU 是一个非线性函数：output = max(0, input)。它把所有的负数变成 0，正数保持不变。
        为什么需要它：如果没有非线性激活函数，无论堆叠多少层 nn.Linear，最终整个网络本质上还是一个线性模型，表达能力有限。ReLU 引入的非线性是神经网络能够学习复杂模式（如识别图像、理解语言）的关键。
        另一种写法：有时人们不把它定义为属性，而是在 forward 中直接使用 F.relu(x)（import torch.nn.functional as F），这两种方式等价。
        """
        self.relu = nn.ReLU()        

        # 全连接层2 (输出层)
        """
        含义：定义输出层，也是一个全连接层。
        参数：输入特征维度是 hidden_size（接收上一层的输出），输出特征维度是 num_classes（我们想要的类别分数）。
        """
        

        
        self.fc2 = nn.Linear(hidden_size, num_classes)  

    # 定义数据如何向前传播
    def forward(self, x):
        """
        含义：定义数据向前传播的计算逻辑。这是 nn.Module 要求你必须重写的方法。
        调用方式：你永远不会直接调用 model.forward(x)。而是直接调用模型实例 model(x)，背后的魔法在于 nn.Module 的 __call__ 方法会为你调用 forward。
        参数 x：网络的输入。通常是形状为 (batch_size, input_size) 的张量。
        """

        
        """
        out
        这是 forward 函数的内部流程：
        self.fc1(x)：输入 x 先经过第一个全连接层，输出一个形状为 (batch_size, hidden_size) 的结果。
        self.relu(out)：然后经过 ReLU 激活函数，输出形状不变，但所有负值变为0。这一步引入了非线性。
        self.fc2(out)：最后经过第二个全连接层（输出层），输出形状为 (batch_size, num_classes) 的原始分数（也称为 logits）。
        最终的 return out：将 logits 返回给调用者。通常，你会在模型外部应用 Softmax 函数将这些 logits 转换为概率分布，但更常见的是将 logits 直接送入 nn.CrossEntropyLoss 损失函数，它内部会高效地结合 Softmax 和交叉熵计算。
        """

        
        out = self.fc1(x)      # x -> fc1
        out = self.relu(out)   # 应用激活函数
        out = self.fc2(out)    # 得到最终输出
        return out

# 实例化网络
model = SimpleNet(input_size=784, hidden_size=128, num_classes=10)
print(model)

In [8]:
import torch.optim as optim

# --- 1. 准备数据 (以MNIST手写数字为例) ---
from torchvision import datasets, transforms

# 数据预处理：转为Tensor并归一化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 下载并加载MNIST训练集
trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# --- 2. 初始化模型、损失函数和优化器 ---
model = SimpleNet(input_size=784, hidden_size=128, num_classes=10)
criterion = nn.CrossEntropyLoss()           # 分类问题用交叉熵损失
optimizer = optim.SGD(model.parameters(), lr=0.01)  # 随机梯度下降优化器

# --- 3. 训练循环 ---
epochs = 5
for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in trainloader:
        # 将28x28的图片展平成784维的向量
        images = images.view(images.size(0), -1)

        # 梯度清零 (否则梯度会累积)
        optimizer.zero_grad()

        # 前向传播、计算损失、反向传播、更新权重
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    print(f'Epoch {epoch+1}, Loss: {running_loss / len(trainloader):.4f}')

print("训练完成！")

Failed to download (trying next):
HTTP Error 404: Not Found

Failed to download (trying next):
<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>



RuntimeError: Error downloading train-images-idx3-ubyte.gz